# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

---

**My answer — ranking / scoring, powered by a classifier.**

My lane (Lane 2, Refresh / Content Opportunity Scoring) is a **ranking** task: the deliverable is an ordered queue of pages, highest refresh priority first, with a score, a suggested action, and reason codes per page. Under the hood, a binary **classifier** estimates each page's probability of being decline-labelled, and that probability (blended with visibility, so a declining page nobody sees does not outrank a declining page everybody sees) becomes the ranking score.

Why not the other types? Not pure classification: a yes/no label with a fixed threshold does not allocate a fixed weekly review budget — the editor needs an *order*, not a verdict on all 30,000 pages. Not clustering: I have an observed outcome to rank against, so unsupervised grouping would throw away the most useful signal I have (clustering answers a different lane's question). The code cell below shows what the output actually looks like: the top of a queue built from one simple transparent score.


In [1]:
import os
import numpy as np
import pandas as pd

# Find the repo root from wherever this kernel started (works locally and in Colab).
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Working dir:", os.getcwd())
print("Rows x columns:", df.shape)
print("Pseudonymized clients:", df["client_id"].nunique())
print("Content types:", df["content_type"].nunique())
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)  # the label, derived in the open
print("Declining pages (trend_direction == 'down'):", int(df["is_declining_label"].sum()))

# What the deliverable looks like: a ranked queue (here: one simple transparent score).
df["demo_score"] = (df["days_since_last_update"] >= 180).astype(int) * np.log1p(df["impressions_90d"])
queue = df.sort_values("demo_score", ascending=False)
cols = ["content_id", "client_id", "impressions_90d", "days_since_last_update",
        "avg_position", "word_count", "is_declining_label"]
print("Base rate (share of all pages that are decline-labelled):", round(df["is_declining_label"].mean(), 3))
print()
print("Top 10 of a simple stale x visible queue:")
print(queue[cols].head(10).to_string(index=False))


Working dir: /home/hatch/workspace/flyrank-ml
Rows x columns: (30000, 44)
Pseudonymized clients: 32
Content types: 3
Declining pages (trend_direction == 'down'): 16262
Base rate (share of all pages that are decline-labelled): 0.542

Top 10 of a simple stale x visible queue:
          content_id         client_id  impressions_90d  days_since_last_update  avg_position  word_count  is_declining_label
content_cf56e2e2e282 client_7f2253d7e2            61678                     194          19.7      5125.0                   1
content_7368877ea310 client_7f2253d7e2            59472                     194          24.8      2591.0                   1
content_1bfaa38ff26c client_7f2253d7e2            25715                     194          22.2      3861.0                   1
content_0a91db491d14 client_7f2253d7e2            13299                     193          10.5      3478.0                   1
content_5feee3994adb client_7f2253d7e2             7812                     194          39.0  

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

---

**My answer.**

**The target:** `is_declining_label` — 1 when a page is decline-labelled, 0 otherwise. I would score every page with its estimated probability of carrying that label.

**Where the label comes from:** it is *derived from observed metrics, not assigned by a person or a product rule*. `trend_direction` is computed from `trend_pct`, the measured percentage change between the page's previous-30-day and last-30-day observed windows; the label is 1 exactly when `trend_direction == "down"`. The dataset deliberately ships no product decision flags (no "needs refresh" score to imitate), so the label reflects measured behaviour of the page, not someone's opinion about it. The code cell below verifies the construction holds on all 30,000 rows.

**The honest caveats, stated now so nobody has to dig for them:** (1) this is a *derived* label — it summarises an observed window rather than recording a future event, so my model learns "looks like pages that declined in this window", and I will phrase claims that way; with the warehouse release, the same framing can be rebuilt with a later window as a genuinely future outcome. (2) Because the label's parents are `trend_direction` and `trend_pct`, those two columns are banned as features forever — using them would be leakage, and my ML-05 leakage work exists precisely to keep that line bright.


In [2]:
# Verify the label construction on every row.
rebuilt = (df["trend_direction"] == "down").astype(int)
print("Rows where rebuilt label == is_declining_label:",
      f"{(rebuilt == df['is_declining_label']).mean():.1%} of {len(df):,}")
print()
print("trend_direction distribution (the label is 1 only for 'down'):")
print(df["trend_direction"].value_counts().to_string())
print()
print("Label base rate:", round(df["is_declining_label"].mean(), 3))
print("Safe features come from the observed metrics (impressions, clicks, position, CTR,")
print("age, freshness, depth...). trend_direction and trend_pct are the label's parents -")
print("they are never in the feature list.")


Rows where rebuilt label == is_declining_label: 100.0% of 30,000

trend_direction distribution (the label is 1 only for 'down'):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

Label base rate: 0.542
Safe features come from the observed metrics (impressions, clicks, position, CTR,
age, freshness, depth...). trend_direction and trend_pct are the label's parents -
they are never in the feature list.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

---

**My answer — Precision@50, measured on a client holdout.**

**The metric:** of the top 50 pages my queue recommends, how many are actually decline-labelled. K = 50 because that is a realistic weekly review batch for an editor — the metric mirrors the real unit of work, and it grades exactly the part of the ranking anyone will ever reach.

**How it must be measured:** on a **client holdout** — whole clients held back from training — because the queue will be used on clients the model has not seen. Row-random splits flatter the score (pages from the same client share too much).

**What number means "good":** context is everything. The overall base rate is 0.542, but on unseen clients the shipped rule baseline only reaches Precision@50 = 0.240, while the reference pipeline's random forest reaches 0.740 — roughly a 3x lift over the baseline. So "good" for my lane = clearly and honestly above the rule baseline on held-out clients, with ROC-AUC reported alongside as a threshold-free sanity check. The code cell below is my own quick version of that measurement: a small random forest on 16 leakage-safe raw features, one fixed seed, one client holdout. It lands at Precision@50 = 0.76 vs 0.42 for a simple two-signal blend on the *same* holdout — the metric is computable today, before any serious modelling.


In [3]:
# Success metric, computed today: Precision@50 on a client holdout (seed fixed at 42).
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k=50):
    top = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[top].mean()

features = ["search_volume", "competition", "cpc", "word_count", "char_count",
            "impressions_90d", "clicks_90d", "sessions_90d", "content_age_days",
            "days_since_last_update", "ctr", "avg_position", "engagement_rate",
            "scroll_rate", "ai_traffic_pct", "days_with_impressions"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

clients = df["client_id"].unique()
rng = np.random.RandomState(42)
test_clients = set(rng.permutation(clients)[:max(1, int(round(len(clients) * 0.2)))])
test_mask = df["client_id"].isin(test_clients).to_numpy()

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X[~test_mask], y[~test_mask])
proba = rf.predict_proba(X[test_mask])[:, 1]

blend = (0.4 * np.log1p(df["impressions_90d"]).rank(pct=True)
         + 0.3 * df["days_since_last_update"].rank(pct=True))

print(f"Holdout: {test_mask.sum():,} pages from {len(test_clients)} unseen clients "
      f"(their base rate: {y[test_mask].mean():.3f})")
print(f"Simple rule blend  -> Precision@50 = {precision_at_k(blend[test_mask], y[test_mask]):.2f}")
print(f"Random forest      -> Precision@50 = {precision_at_k(proba, y[test_mask]):.2f}, "
      f"ROC-AUC = {roc_auc_score(y[test_mask], proba):.3f}")
print()
print("Reference point (repo's shipped pipeline, richer features, same client-holdout idea):")
print("rule baseline Precision@50 = 0.240, random forest = 0.740.")


Holdout: 3,381 pages from 6 unseen clients (their base rate: 0.525)
Simple rule blend  -> Precision@50 = 0.42
Random forest      -> Precision@50 = 0.76, ROC-AUC = 0.677

Reference point (repo's shipped pipeline, richer features, same client-holdout idea):
rule baseline Precision@50 = 0.240, random forest = 0.740.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

---

**My answer.**

**One row = one content page** belonging to one pseudonymized client, summarised over the trailing 90 days: its search visibility (impressions, clicks, average position, CTR), its engagement (sessions, scroll, engagement rate), its content metadata (type, intent, word count), and its age/freshness fields. The grain check in the code below confirms `content_id` is unique across all 30,000 rows — no page appears twice, so one row really is one page.

**My lane's working slice:** pages with observed visibility — `impressions_90d >= 100`, which is 22,006 of the 30,000 pages. A refresh queue can only prioritise on observed evidence; pages with essentially no impressions are a different problem (they are not being found at all) and would flood the queue with noise if I let them in. The full dataset stays my training population for the label; the slice is the population the queue is *about*. The dataframe below is that slice, showing the columns an editor's queue would carry.


In [4]:
# Grain check + the lane's working slice as a real dataframe.
print("Grain check - one row per content page:")
print(f"  rows: {len(df):,} | unique content_id: {df['content_id'].nunique():,} "
      f"| duplicated content_id: {int(df.duplicated('content_id').sum())}")
print(f"  clients: {df['client_id'].nunique()} | rows per client: "
      f"{len(df) / df['client_id'].nunique():,.0f} on average")

lane = df[df["impressions_90d"] >= 100].copy()
print(f"\nLane slice (visible pages, impressions_90d >= 100): {len(lane):,} rows")
print(f"Decline-labelled share inside the slice: {lane['is_declining_label'].mean():.1%}")
cols = ["content_id", "client_id", "content_type", "impressions_90d", "clicks_90d",
        "avg_position", "ctr", "word_count", "days_since_last_update", "is_declining_label"]
print()
print(lane[cols].head(8).to_string(index=False))


Grain check - one row per content page:
  rows: 30,000 | unique content_id: 30,000 | duplicated content_id: 0
  clients: 32 | rows per client: 938 on average

Lane slice (visible pages, impressions_90d >= 100): 22,006 rows
Decline-labelled share inside the slice: 59.8%

          content_id         client_id    content_type  impressions_90d  clicks_90d  avg_position  ctr  word_count  days_since_last_update  is_declining_label
content_304f48230142 client_f369cb89fc keyword article             3803          29          10.6 0.76      3221.0                      20                   1
content_a1fb4e703a9e client_4e07408562 keyword article            15320           7          20.3 0.05      2481.0                      25                   1
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581          11          36.5 0.09      3515.0                      20                   1
content_331d6c4de07b client_19581e27de keyword article            11751          58          

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

---

**My answer — three measured reasons a fixed rule is not enough:**

1. **The headline rule barely has anything to act on.** "Review stale pages" sounds sensible, but only 174 of 30,000 pages are 180+ days un-updated — and 51.1% of pages updated within the last 30 days are declining anyway. Decline in this data is mostly *not* a staleness story, so a staleness if-statement structurally misses the problem.
2. **Single signals collapse on unseen clients.** On my client holdout (section 3), a sensible two-signal rule blend scored Precision@50 = 0.42 and the repo's shipped rule baseline scores 0.240, while learned models reach 0.74–0.76. Rules that look fine on the full data do not travel to new clients; the combinations do.
3. **The signals interact.** In the table below, the measured decline rate across freshness x visibility tiers runs from about 0.38 to 0.66 in the well-populated cells, and the direction of the visibility effect flips between freshness bands — moderate-visibility pages decline *more* than low-visibility ones among fresh pages, while among 91–180-day pages the excellent-visibility tier declines *least*. There is no single threshold on any one column that reproduces that surface. A tree ensemble earns its place precisely because it learns combinations of signals instead of one hand-set cut.

None of this makes the rule baseline useless — it stays in my project as the transparent yardstick every model has to beat, and its reason codes are how the queue explains itself to an editor.


In [5]:
# The interaction a fixed rule cannot write: measured decline rate by freshness x visibility.
rates = df.pivot_table(index="freshness_tier", columns="impression_tier",
                       values="is_declining_label", aggfunc="mean").round(3)
counts = df.pivot_table(index="freshness_tier", columns="impression_tier",
                        values="is_declining_label", aggfunc="count")
print("Decline rate by freshness_tier x impression_tier:")
print(rates.to_string())
print()
print("Rows behind each cell (small cells = read with care):")
print(counts.to_string())
print()
fresh = df[df["freshness_tier"] == "0-30"]
print(f"And the stale-rule problem in one line: {len(fresh):,} pages were updated in the last")
print(f"30 days, yet {fresh['is_declining_label'].mean():.1%} of them are measured declining -")
print(f"while only {(df['days_since_last_update'] >= 180).sum()} pages in the whole dataset are 180+ days stale.")


Decline rate by freshness_tier x impression_tier:
impression_tier  excellent   good    low  moderate
freshness_tier                                    
0-30                 0.485  0.578  0.424     0.591
181+                 1.000  1.000  0.421     0.692
31-90                0.429  0.381  0.655     0.609
91-180               0.433  0.598  0.590     0.656

Rows behind each cell (small cells = read with care):
impression_tier  excellent  good   low  moderate
freshness_tier                                  
0-30                   582  4149  9082      6667
181+                     2     7   152        13
31-90                    7    21    55        92
91-180                 487  3028  1959      3697

And the stale-rule problem in one line: 20,480 pages were updated in the last
30 days, yet 51.1% of them are measured declining -
while only 174 pages in the whole dataset are 180+ days stale.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.